In [1]:
cd "C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph"

C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph


#  Agent 1:TOPIC AGENT

In [47]:
%%writefile src/agents/topic_agent.py

import sys

from langchain_core.runnables import RunnableLambda

from src.llm.llm import MeetingLLM
from src.llm.output_parser import OutputParser
from src.prompts.topic_prompt import TOPIC_PROMPT
from src.utils.exception import ProjectException
from src.utils.logger import get_logger

logger = get_logger(__name__)


class TopicAgent:
    

    QUERY = "What are the main discussion topics in this meeting?"

    @classmethod
    def invoke(cls, retriever):


        try:
            llm = MeetingLLM.load_model()
            parser = OutputParser.topic_parser()

            def retrieve_context(_):
                results = retriever.retrieve(cls.QUERY)

                return "\n\n".join(
                    node.text for node in results
                )

            chain = (
              RunnableLambda(
                 lambda _: {
                     "transcript": retrieve_context(None),
                     "format_instructions": parser.get_format_instructions(),})
                | TOPIC_PROMPT
                | llm
                | parser
            )

            logger.info("Topic Agent executed successfully.")

            return chain.invoke({})

        except Exception as error:
            logger.error(str(error))
            raise ProjectException(str(error), sys)

Overwriting src/agents/topic_agent.py


In [48]:
# test for Topic Agent 

In [49]:
%%writefile tests/test_topic_agent.py

from pathlib import Path

from src.agents.topic_agent import TopicAgent
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker
from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever


RAW_PATH = Path("data/raw/Meeting_Transcript.txt")


def build_retriever():
    document = TranscriptLoader.load_document(str(RAW_PATH))
    clean_document = TranscriptCleaner.clean(document)
    nodes = TranscriptChunker.create_nodes(clean_document)

    embedding_model = EmbeddingModel.load_model()

    vector_index = FAISSIndexManager.create_index(
        nodes=nodes,
        embedding_model=embedding_model,
    )

    return TranscriptRetriever.create_retriever(vector_index)


def test_topic_agent_runs():

    retriever = build_retriever()

    output = TopicAgent.invoke(retriever)

    assert output is not None


def test_topic_output_contains_topics():

    retriever = build_retriever()

    output = TopicAgent.invoke(retriever)

    assert len(output.topics) > 0

Overwriting tests/test_topic_agent.py


In [50]:
import importlib
import src.prompts.topic_prompt as topic_prompt

importlib.reload(topic_prompt)

TOPIC_PROMPT = topic_prompt.TOPIC_PROMPT

print("Topic prompt reloaded successfully!")

Topic prompt reloaded successfully!


In [51]:
import sys

!{sys.executable} -m pytest tests/test_topic_agent.py -v


============================= test session starts =============================
platform win32 -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Lenovo\anaconda3\envs\meetingnotes\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
configfile: pytest.ini
plugins: anyio-4.12.1, langsmith-0.12.2
collecting ... collected 2 items

tests/test_topic_agent.py::test_topic_agent_runs PASSED                  [ 50%]
tests/test_topic_agent.py::test_topic_output_contains_topics PASSED      [100%]

======================== 2 passed in 316.19s (0:05:16) ========================


# Imports

In [52]:
from pathlib import Path

from src.config.config import load_config
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker
from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever

from src.agents.topic_agent import TopicAgent

# Build Retriever

# Display Topics

In [54]:
import importlib

import src.agents.topic_agent as topic_agent

# Reload the updated file
importlib.reload(topic_agent)

# Import the refreshed class
TopicAgent = topic_agent.TopicAgent

print("✅ TopicAgent reloaded successfully.")

✅ TopicAgent reloaded successfully.


In [55]:
from pathlib import Path

# Configuration
from src.config.config import load_config

# Data Pipeline
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker

# Vector Store
from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever

# Agent
from src.agents.topic_agent import TopicAgent

# ----------------------------
# Load Configuration
# ----------------------------
config = load_config()

TRANSCRIPT_PATH = Path(config["paths"]["raw_data"]) / "Meeting_Transcript.txt"

# ----------------------------
# Load & Preprocess Transcript
# ----------------------------
document = TranscriptLoader.load_document(str(TRANSCRIPT_PATH))
clean_document = TranscriptCleaner.clean(document)

# ----------------------------
# Create Chunks
# ----------------------------
nodes = TranscriptChunker.create_nodes(clean_document)

# ----------------------------
# Create Embeddings & FAISS Index
# ----------------------------
embedding_model = EmbeddingModel.load_model()

vector_index = FAISSIndexManager.create_index(
    nodes=nodes,
    embedding_model=embedding_model,
)

# ----------------------------
# Create Retriever
# ----------------------------
retriever = TranscriptRetriever.create_retriever(vector_index)
#
# ----------------------------
# Run Topic Agent
# ----------------------------
topic_output = TopicAgent.invoke(retriever)

# ----------------------------
# Display Output
# ----------------------------
print("=" * 50)
print("📌 MAIN DISCUSSION TOPICS")
print("=" * 50)

for i, topic in enumerate(topic_output.topics, start=1):
    print(f"{i}. {topic}")

2026-09-09 15:18:13 | INFO | src.data_ingestion.loader | Transcript loaded successfully: Meeting_Transcript.txt
2026-09-09 15:18:13 | INFO | src.preprocessing.cleaner | Transcript Normalized Sucessfully
2026-09-09 15:18:13 | INFO | src.preprocessing.chunker | Created 2 transcript chunks.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-09-09 15:18:20 | INFO | src.vector_store.embedding_model | Embedding model loaded: BAAI/bge-small-en-v1.5
2026-09-09 15:18:20 | INFO | src.vector_store.faiss_index | FAISS vector index created successfully.
2026-09-09 15:18:20 | INFO | src.vector_store.retriever | Retriever created with Top-K = 3
2026-09-09 15:18:29 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-09 15:18:29 | INFO | src.agents.topic_agent | Topic Agent executed successfully.


📌 MAIN DISCUSSION TOPICS
1. login issue
2. API response format
3. error handling in mobile app
4. authentication API changes
5. device-specific login issues


# Agent 2 :Summary Agent

In [69]:
%%writefile src/agents/summary_agent.py

import sys

from langchain_core.runnables import RunnableLambda

from src.llm.llm import MeetingLLM
from src.llm.output_parser import OutputParser
from src.prompts.summary_prompt import SUMMARY_PROMPT
from src.utils.exception import ProjectException
from src.utils.logger import get_logger

logger = get_logger(__name__)

class SummaryAgent:
    QUERY = ("Summarize this meeting including objective, key discussions, and decisions.")

    @classmethod
    def invoke(cls,retriever):

        try:
            llm=MeetingLLM.load_model()
            parser=OutputParser.summary_parser()

            def retrieve_context(_):
                results=retriever.retrieve(cls.QUERY)
                return "\n\n".join(node.text for node in results)

            chain=(RunnableLambda(lambda _:{"transcript": retrieve_context(None),
                        "format_instructions": parser.get_format_instructions(),})
                       |SUMMARY_PROMPT|llm|parser)

            logger.info("Summary Agent executed successfully.")
            
            return chain.invoke({})

        except Exception as error:
            logger.error(str(error))
            raise ProjectException(str(error), sys)

                       



Overwriting src/agents/summary_agent.py


# Testing SummaryAgent

In [70]:
%%writefile tests/test_summary_agent.py

from pathlib import Path

from src.agents.summary_agent import SummaryAgent
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker
from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever


RAW_PATH = Path("data/raw/Meeting_Transcript.txt")


def build_retriever():
    document = TranscriptLoader.load_document(str(RAW_PATH))
    clean_document = TranscriptCleaner.clean(document)
    nodes = TranscriptChunker.create_nodes(clean_document)

    embedding_model = EmbeddingModel.load_model()

    vector_index = FAISSIndexManager.create_index(
        nodes=nodes,
        embedding_model=embedding_model,
    )

    return TranscriptRetriever.create_retriever(vector_index)


def test_summary_agent_runs():
    retriever = build_retriever()

    output = SummaryAgent.invoke(retriever)

    assert output is not None


def test_summary_output_fields():
    retriever = build_retriever()

    output = SummaryAgent.invoke(retriever)

    assert output.meeting_objective != ""
    assert len(output.key_discussion_points) > 0
    assert isinstance(output.decisions_taken, list)

Overwriting tests/test_summary_agent.py


In [71]:
import sys

!{sys.executable} -m pytest tests/test_summary_agent.py

============================= test session starts =============================
platform win32 -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
configfile: pytest.ini
plugins: anyio-4.12.1, langsmith-0.12.2
collected 2 items

tests\test_summary_agent.py ..                                           [100%]

======================== 2 passed in 225.24s (0:03:45) ========================


In [72]:
from src.agents.summary_agent import SummaryAgent
summary_output = SummaryAgent.invoke(retriever)

summary_output

2026-09-09 15:39:42 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-09 15:39:42 | INFO | src.agents.summary_agent | Summary Agent executed successfully.


SummaryOutput(meeting_objective='Identify and resolve the mobile app crashing issue during login.', key_discussion_points=['The root cause of the crash is related to authentication API changes deployed three days ago.', 'A detailed bug report with screenshots and logs has been prepared by QA Tester.', 'Mobile Developer reviewed Android login module for error handling, while Backend Developer verified authentication API for breaking changes.'], decisions_taken=['QA Tester will share the bug report within the next hour.', 'Mobile Developer will start working on the fix immediately after this meeting.', 'Backend Developer will review server logs right away to identify any authentication errors recorded.'])

# Agent 3:Action Item Agent

In [81]:
%%writefile src/agents/action_agent.py


import sys

from langchain_core.runnables import RunnableLambda

from src.llm.llm import MeetingLLM
from src.llm.output_parser import OutputParser
from src.prompts.action_prompt import ACTION_PROMPT
from src.utils.exception import ProjectException
from src.utils.logger import get_logger

logger = get_logger(__name__)


class ActionAgent:

    QUERY=("Extract all action items, task owners, and deadlines from this meeting.")

    @classmethod
    def invoke(cls,retriever):
        try:
            llm = MeetingLLM.load_model()
            parser = OutputParser.action_parser()

            def retrieve_context(_):
                results = retriever.retrieve(cls.QUERY)

                return "\n\n".join(node.text for node in results)

            chain = (
                RunnableLambda(
                    lambda _: {
                        "transcript": retrieve_context(None),
                        "format_instructions": parser.get_format_instructions(),})| ACTION_PROMPT| llm| parser)

            logger.info("Action Agent executed successfully.")

            return chain.invoke({})

        except Exception as error:
            logger.error(str(error))
            raise ProjectException(str(error), sys)
    


Overwriting src/agents/action_agent.py


# Testing Action Agent 


In [82]:
%%writefile tests/test_action_agent.py

from pathlib import Path

from src.agents.action_agent import ActionAgent
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker
from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever


RAW_PATH = Path("data/raw/Meeting_Transcript.txt")


def build_retriever():
    document = TranscriptLoader.load_document(str(RAW_PATH))
    clean_document = TranscriptCleaner.clean(document)
    nodes = TranscriptChunker.create_nodes(clean_document)

    embedding_model = EmbeddingModel.load_model()

    vector_index = FAISSIndexManager.create_index(
        nodes=nodes,
        embedding_model=embedding_model,
    )

    return TranscriptRetriever.create_retriever(vector_index)


def test_action_agent_runs():
    retriever = build_retriever()

    output = ActionAgent.invoke(retriever)

    assert output is not None


def test_action_output_contains_items():
    retriever = build_retriever()

    output = ActionAgent.invoke(retriever)

    assert len(output.action_items) > 0


def test_action_item_fields():
    retriever = build_retriever()

    output = ActionAgent.invoke(retriever)

    item = output.action_items[0]

    assert item.task != ""
    assert item.owner != ""
    assert isinstance(item.deadline, str)

Overwriting tests/test_action_agent.py


In [83]:
import importlib
import src.prompts.action_prompt as action_prompt

importlib.reload(action_prompt)

<module 'src.prompts.action_prompt' from 'C:\\Users\\Lenovo\\Desktop\\github projects\\ai-meeting-notes-analyzer-langgraph\\src\\prompts\\action_prompt.py'>

In [84]:
import sys

!{sys.executable} -m pytest tests/test_action_agent.py

============================= test session starts =============================
platform win32 -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
configfile: pytest.ini
plugins: anyio-4.12.1, langsmith-0.12.2
collected 3 items

tests\test_action_agent.py ...                                           [100%]

======================== 3 passed in 305.21s (0:05:05) ========================


In [85]:
from src.agents.action_agent import ActionAgent
action_output = ActionAgent.invoke(retriever)

action_output

2026-09-09 16:09:00 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-09 16:09:00 | INFO | src.agents.action_agent | Action Agent executed successfully.


ActionOutput(action_items=[ActionItem(task='Review the Android login module and check if any error handling is missing.', owner='Mobile Developer', deadline='Not Mentioned'), ActionItem(task='Prepare a detailed bug report with screenshots and logs.', owner='QA Tester', deadline='Not Mentioned'), ActionItem(task='Verify the authentication API and check if there are any breaking changes.', owner='Backend Developer', deadline='Not Mentioned')])

# Agent4 :Priority Classification Agent

In [86]:
%%writefile src/agents/priority_agent.py


import sys

from langchain_core.runnables import RunnableLambda

from src.llm.llm import MeetingLLM
from src.llm.output_parser import OutputParser
from src.prompts.priority_prompt import PRIORITY_PROMPT
from src.utils.exception import ProjectException
from src.utils.logger import get_logger

logger = get_logger(__name__)


class PriorityAgent:

    QUERY = (
        "Identify all tasks discussed in this meeting and classify "
        "their priority as High, Medium, or Low."
    )

    @classmethod
    def invoke(cls, retriever):

        try:
            llm = MeetingLLM.load_model()
            parser = OutputParser.priority_parser()

            def retrieve_context(_):
                results = retriever.retrieve(cls.QUERY)

                return "\n\n".join(node.text for node in results)

            chain = (RunnableLambda(lambda _: {
                        "transcript": retrieve_context(None),
                        "format_instructions": parser.get_format_instructions(),})| PRIORITY_PROMPT| llm| parser)

            logger.info("Priority Agent executed successfully.")

            return chain.invoke({})

        except Exception as error:
            logger.error(str(error))
            raise ProjectException(str(error), sys)

Writing src/agents/priority_agent.py


In [89]:
import importlib
import src.prompts.priority_prompt as priority_prompt

importlib.reload(priority_prompt)

<module 'src.prompts.priority_prompt' from 'C:\\Users\\Lenovo\\Desktop\\github projects\\ai-meeting-notes-analyzer-langgraph\\src\\prompts\\priority_prompt.py'>

# test_priority_agent

In [90]:
%%writefile tests/test_priority_agent.py

from pathlib import Path

from src.agents.priority_agent import PriorityAgent
from src.data_ingestion.loader import TranscriptLoader
from src.preprocessing.cleaner import TranscriptCleaner
from src.preprocessing.chunker import TranscriptChunker
from src.vector_store.embedding_model import EmbeddingModel
from src.vector_store.faiss_index import FAISSIndexManager
from src.vector_store.retriever import TranscriptRetriever


RAW_PATH = Path("data/raw/Meeting_Transcript.txt")


def build_retriever():
    document = TranscriptLoader.load_document(str(RAW_PATH))
    clean_document = TranscriptCleaner.clean(document)
    nodes = TranscriptChunker.create_nodes(clean_document)

    embedding_model = EmbeddingModel.load_model()

    vector_index = FAISSIndexManager.create_index(
        nodes=nodes,
        embedding_model=embedding_model,
    )

    return TranscriptRetriever.create_retriever(vector_index)


def test_priority_agent_runs():
    retriever = build_retriever()

    output = PriorityAgent.invoke(retriever)

    assert output is not None


def test_priority_output_contains_items():
    retriever = build_retriever()

    output = PriorityAgent.invoke(retriever)

    assert len(output.priorities) > 0


def test_priority_item_fields():
    retriever = build_retriever()

    output = PriorityAgent.invoke(retriever)

    item = output.priorities[0]

    assert item.task != ""
    assert item.priority in ["High", "Medium", "Low"]

Overwriting tests/test_priority_agent.py


In [91]:
import sys

!{sys.executable} -m pytest tests/test_priority_agent.py

============================= test session starts =============================
platform win32 -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
configfile: pytest.ini
plugins: anyio-4.12.1, langsmith-0.12.2
collected 3 items

tests\test_priority_agent.py ...                                         [100%]

======================== 3 passed in 287.26s (0:04:47) ========================


In [92]:
from src.agents.priority_agent import PriorityAgent
priority_output = PriorityAgent.invoke(retriever)

priority_output

2026-09-09 16:32:12 | INFO | src.llm.llm | Loaded LLM: Qwen2.5-3B-Instruct.gguf
2026-09-09 16:32:12 | INFO | src.agents.priority_agent | Priority Agent executed successfully.


PriorityOutput(priorities=[PriorityItem(task='Review the issue tracker with all findings.', priority='High'), PriorityItem(task='Perform regression testing after the fix is ready.', priority='Medium'), PriorityItem(task='Add better error handling to prevent app crashes even if API fails.', priority='Low')])